# VeriNews AI — Milestone 2: DistilBERT Fine-Tuning (Colab/Kaggle GPU)

> **Important:** This notebook is designed for **GPU-accelerated environments only** (Google Colab T4/A100, Kaggle P100/T4).  
> The `train_distilbert.py` script will refuse to run full training on CPU.  
> For a local smoke test, see the README.

## Prerequisites
1. GPU runtime selected (Colab: Runtime → Change runtime type → T4 GPU)
2. `unified_dataset.csv` available — either from the repo or uploaded manually
3. (Optional) Google Drive mounted for saving checkpoints

## What this notebook does
1. Clones the VeriNews AI repository
2. Installs dependencies
3. Generates frozen train/val/test splits with provenance manifest
4. Runs token-length analysis to confirm `MAX_SEQ_LEN`
5. Runs full DistilBERT fine-tuning (3 epochs)
6. Evaluates on the frozen test split and the 60-example manual validation set
7. Saves the checkpoint to Google Drive

## Step 0 — Verify GPU

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected. Change runtime to GPU before proceeding.')

## Step 1 — Clone repository

In [ ]:
import os

REPO_URL    = 'https://github.com/YOUR-USERNAME/VeriNews-AI.git'  # ← update this
REPO_BRANCH = 'main'
REPO_DIR    = '/content/VeriNews-AI'

if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(f'{REPO_DIR}/backend')
print(f'Working directory: {os.getcwd()}')

## Step 2 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## Step 3 — Upload dataset

The `datasets/` directory is excluded from git (large files).  
Upload or copy `unified_dataset.csv` to `backend/datasets/processed/` now.

In [ ]:
import os
from pathlib import Path

DATASET_PATH = Path('datasets/processed/unified_dataset.csv')

if not DATASET_PATH.exists():
    # Option A: Upload from local machine
    from google.colab import files
    DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
    print('Upload unified_dataset.csv now:')
    uploaded = files.upload()
    for name, data in uploaded.items():
        DATASET_PATH.write_bytes(data)
    print(f'Saved to {DATASET_PATH}')
    
    # Option B: Copy from Google Drive (uncomment if preferred)
    # from google.colab import drive
    # drive.mount('/content/drive')
    # !cp '/content/drive/MyDrive/VeriNews/unified_dataset.csv' {DATASET_PATH}
else:
    print(f'Dataset already present: {DATASET_PATH} ({DATASET_PATH.stat().st_size/1e6:.1f} MB)')

## Step 4 — Generate frozen splits + token-length analysis

In [ ]:
!python train/transformer/dataset.py --prepare --analyze-lengths --sample-size 15000

## Step 5 — Review token-length stats and confirm MAX_SEQ_LEN

Read the output above.  
Choose `MAX_SEQ_LEN` based on the p90/p95 percentiles and the truncation trade-off guide.  
Update the value below before running Step 6.

In [ ]:
import json

stats_path = 'results/distilbert_v1/token_length_stats.json'
with open(stats_path) as f:
    stats = json.load(f)

print('Token-length statistics:')
for k, v in stats.items():
    if k != 'notes':
        print(f'  {k:<14}: {v}')

# ── SET THIS VALUE AFTER REVIEWING THE OUTPUT ABOVE ──────────────────────────
MAX_SEQ_LEN = 256  # ← update based on p90/p95 from Step 4 output
print(f'\nSelected MAX_SEQ_LEN: {MAX_SEQ_LEN}')
print('This value will be passed to train_distilbert.py --max-seq-len')

## Step 6 — Full training (GPU required)

In [ ]:
!python train/transformer/train_distilbert.py \
    --max-seq-len {MAX_SEQ_LEN} \
    --epochs 3 \
    --batch-size 16 \
    --lr 2e-5 \
    --weight-decay 0.01

## Step 7 — Evaluate

In [ ]:
!python train/transformer/evaluate_distilbert.py \
    --checkpoint saved_models/distilbert_v1/best \
    --max-seq-len {MAX_SEQ_LEN}

## Step 8 — Save checkpoint to Google Drive

In [ ]:
# Mount Drive if not already mounted
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil, os
DRIVE_DEST = '/content/drive/MyDrive/VeriNews/distilbert_v1'
os.makedirs(DRIVE_DEST, exist_ok=True)

# Copy checkpoint
shutil.copytree('saved_models/distilbert_v1/best', f'{DRIVE_DEST}/best', dirs_exist_ok=True)
# Copy results
shutil.copytree('results/distilbert_v1', f'{DRIVE_DEST}/results', dirs_exist_ok=True)
# Copy split manifests
shutil.copytree('datasets/processed/distilbert_v1', f'{DRIVE_DEST}/splits', dirs_exist_ok=True)

print(f'Checkpoint and results saved to {DRIVE_DEST}')

## Step 9 — Display key results

In [ ]:
import json
from IPython.display import Image, display

# Test split metrics
with open('results/distilbert_v1/metrics.json') as f:
    metrics = json.load(f)
print('=== Test Split Metrics ===')
for k, v in metrics.items():
    if isinstance(v, (int, float)):
        print(f'  {k:<15}: {v}')

print()

# Manual validation metrics
print('=== Manual Validation Report preview ===')
with open('results/distilbert_v1/manual_validation_distilbert_report.md') as f:
    for i, line in enumerate(f):
        if i < 20:
            print(line, end='')

print()
print('=== Confusion Matrix ===')
display(Image('results/distilbert_v1/confusion_matrix.png'))